# UrbanFlow – Baseline Models

This notebook establishes baseline forecasting models for comparison with
the advanced UrbanFlow architecture.

The models predict traffic speed approximately 30 minutes ahead using the
processed METR-LA traffic sequences created in Notebook 02.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
train_data = np.load("../data/processed/train_sequences.npz")
val_data = np.load("../data/processed/validation_sequences.npz")
test_data = np.load("../data/processed/test_sequences.npz")

X_train = train_data["X"]
y_train = train_data["y"]

X_val = val_data["X"]
y_val = val_data["y"]

X_test = test_data["X"]
y_test = test_data["y"]

print("Training:")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nValidation:")
print("X:", X_val.shape)
print("y:", y_val.shape)

print("\nTest:")
print("X:", X_test.shape)
print("y:", y_test.shape)

Training:
X: (23973, 12, 207)
y: (23973, 207)

Validation:
X: (3410, 12, 207)
y: (3410, 207)

Test:
X: (6838, 12, 207)
y: (6838, 207)


In [3]:
scaler = joblib.load("../data/processed/traffic_scaler.pkl")

print("Scaler loaded successfully.")

Scaler loaded successfully.


In [4]:
NUM_SENSORS = X_train.shape[2]
INPUT_STEPS = X_train.shape[1]

print("Number of sensors:", NUM_SENSORS)
print("Input time steps:", INPUT_STEPS)
print("Historical window:", INPUT_STEPS * 5, "minutes")
print("Forecast horizon:", 30, "minutes")

Number of sensors: 207
Input time steps: 12
Historical window: 60 minutes
Forecast horizon: 30 minutes


In [5]:

y_pred_persistence = X_test[:, -1, :]

print("Persistence prediction shape:", y_pred_persistence.shape)
print("Actual target shape:", y_test.shape)

Persistence prediction shape: (6838, 207)
Actual target shape: (6838, 207)


In [6]:
y_test_original = scaler.inverse_transform(y_test)
y_pred_persistence_original = scaler.inverse_transform(
    y_pred_persistence
)

print("Original target shape:", y_test_original.shape)
print("Original prediction shape:", y_pred_persistence_original.shape)

Original target shape: (6838, 207)
Original prediction shape: (6838, 207)


In [7]:
persistence_mae = mean_absolute_error(
    y_test_original,
    y_pred_persistence_original
)

persistence_rmse = np.sqrt(
    mean_squared_error(
        y_test_original,
        y_pred_persistence_original
    )
)

print("Persistence Baseline")
print("--------------------")
print("MAE :", round(persistence_mae, 4))
print("RMSE:", round(persistence_rmse, 4))

Persistence Baseline
--------------------
MAE : 3.8137
RMSE: 7.6643


In [8]:

X_train_ml = X_train.reshape(X_train.shape[0], -1)
X_val_ml = X_val.reshape(X_val.shape[0], -1)
X_test_ml = X_test.reshape(X_test.shape[0], -1)

print("Training ML input:", X_train_ml.shape)
print("Validation ML input:", X_val_ml.shape)
print("Test ML input:", X_test_ml.shape)

Training ML input: (23973, 2484)
Validation ML input: (3410, 2484)
Test ML input: (6838, 2484)


In [9]:

linear_model = LinearRegression()

linear_model.fit(X_train_ml, y_train)

print("Linear Regression training completed.")

Linear Regression training completed.


In [10]:

y_val_pred_linear = linear_model.predict(X_val_ml)

print("Validation prediction shape:", y_val_pred_linear.shape)
print("Validation target shape:", y_val.shape)

Validation prediction shape: (3410, 207)
Validation target shape: (3410, 207)


In [11]:

y_val_original = scaler.inverse_transform(y_val)

y_val_pred_linear_original = scaler.inverse_transform(
    y_val_pred_linear
)

linear_val_mae = mean_absolute_error(
    y_val_original,
    y_val_pred_linear_original
)

linear_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_original,
        y_val_pred_linear_original
    )
)

print("Linear Regression - Validation")
print("--------------------------------")
print("MAE :", round(linear_val_mae, 4))
print("RMSE:", round(linear_val_rmse, 4))

Linear Regression - Validation
--------------------------------
MAE : 3.8002
RMSE: 6.2791


In [13]:

rf_model = RandomForestRegressor(
    n_estimators=20,
    max_depth=10,
    max_features=0.3,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_ml, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [14]:
y_val_pred_rf = rf_model.predict(X_val_ml)

print("Validation prediction shape:", y_val_pred_rf.shape)
print("Validation target shape:", y_val.shape)

Validation prediction shape: (3410, 207)
Validation target shape: (3410, 207)


In [15]:
y_val_pred_rf_original = scaler.inverse_transform(y_val_pred_rf)

rf_val_mae = mean_absolute_error(
    y_val_original,
    y_val_pred_rf_original
)

rf_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_original,
        y_val_pred_rf_original
    )
)

print("Random Forest - Validation")
print("---------------------------")
print("MAE :", round(rf_val_mae, 4))
print("RMSE:", round(rf_val_rmse, 4))

Random Forest - Validation
---------------------------
MAE : 3.6137
RMSE: 6.6715


In [16]:

y_val_pred_persistence = X_val[:, -1, :]

y_val_pred_persistence_original = scaler.inverse_transform(
    y_val_pred_persistence
)

persistence_val_mae = mean_absolute_error(
    y_val_original,
    y_val_pred_persistence_original
)

persistence_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_original,
        y_val_pred_persistence_original
    )
)

print("Persistence Baseline - Validation")
print("---------------------------")
print("MAE :", round(persistence_val_mae, 4))
print("RMSE:", round(persistence_val_rmse, 4))

Persistence Baseline - Validation
---------------------------
MAE : 3.6178
RMSE: 7.2893


In [21]:

val_target_mask = np.load(
    "../data/processed/validation_target_mask.npy"
)

print("Validation target mask shape:", val_target_mask.shape)
print("Total target values:", val_target_mask.size)
print("Originally observed:", val_target_mask.sum())
print("Originally missing:", (~val_target_mask).sum())
print(
    "Observed percentage:",
    round(100 * val_target_mask.mean(), 2),
    "%"
)

Validation target mask shape: (3410, 207)
Total target values: 705870
Originally observed: 664844
Originally missing: 41026
Observed percentage: 94.19 %


In [22]:

def masked_metrics(y_true, y_pred, mask):
    true_values = y_true[mask]
    pred_values = y_pred[mask]

    mae = mean_absolute_error(true_values, pred_values)
    rmse = np.sqrt(mean_squared_error(true_values, pred_values))

    return mae, rmse


persistence_masked_mae, persistence_masked_rmse = masked_metrics(
    y_val_original,
    y_val_pred_persistence_original,
    val_target_mask
)

linear_masked_mae, linear_masked_rmse = masked_metrics(
    y_val_original,
    y_val_pred_linear_original,
    val_target_mask
)

rf_masked_mae, rf_masked_rmse = masked_metrics(
    y_val_original,
    y_val_pred_rf_original,
    val_target_mask
)


print("Masked Validation Results")
print("-------------------------")
print(
    f"Persistence       | MAE: {persistence_masked_mae:.4f} | "
    f"RMSE: {persistence_masked_rmse:.4f}"
)

print(
    f"Linear Regression | MAE: {linear_masked_mae:.4f} | "
    f"RMSE: {linear_masked_rmse:.4f}"
)

print(
    f"Random Forest     | MAE: {rf_masked_mae:.4f} | "
    f"RMSE: {rf_masked_rmse:.4f}"
)

Masked Validation Results
-------------------------
Persistence       | MAE: 3.8050 | RMSE: 7.4818
Linear Regression | MAE: 3.9204 | RMSE: 6.4007
Random Forest     | MAE: 3.7433 | RMSE: 6.7961


In [23]:

baseline_results = pd.DataFrame({
    "Model": [
        "Persistence",
        "Linear Regression",
        "Random Forest"
    ],
    "Validation_MAE": [
        persistence_masked_mae,
        linear_masked_mae,
        rf_masked_mae
    ],
    "Validation_RMSE": [
        persistence_masked_rmse,
        linear_masked_rmse,
        rf_masked_rmse
    ]
})

baseline_results

,Model,Validation_MAE,Validation_RMSE
0,Persistence,3.804962,7.481793
1,Linear Regression,3.920410,6.400667
2,Random Forest,3.743304,6.796092


In [24]:

import os

os.makedirs("../results", exist_ok=True)

baseline_results.to_csv(
    "../results/baseline_validation_results.csv",
    index=False
)

print("Baseline results saved successfully.")

Baseline results saved successfully.
